# 04 — Supervisor Pattern

**Requires:** `GROQ_API_KEY` in `.env`

---

## What's new here?

Notebook 03 used a **fixed pipeline** — researcher always runs before writer.  
The supervisor pattern is **dynamic** — a central agent decides who runs next.

```
                  ┌──────────────┐
  User Task ───▶  │  SUPERVISOR  │  ◀─────────────────────┐
                  └──────┬───────┘                         │
                         │  routes to...                   │
               ┌─────────┼─────────┐                       │
               ▼         ▼         ▼                       │
          Analyst    Summarizer  Fact-Checker               │
               │         │         │                       │
               └─────────┴─────────┘                       │
                          │ done, report back               │
                          └───────────────────────────── ──┘
                                    (loop until done)
```

**Why this matters:**  
The supervisor can route to agents in **any order**, run the same agent **multiple times**,  
or skip agents entirely — based on what the task actually needs.

**Concepts covered:**
| Concept | What it means |
|---------|---------------|
| Supervisor node | One LLM that reads state and decides who works next |
| Dynamic routing | The path through the graph changes per task |
| Worker → supervisor loop | Workers always return control to the supervisor |
| Structured output | Supervisor returns JSON (`next_worker`) for reliable routing |

## Step 1 — Setup

In [ ]:
# %pip install langgraph langchain langchain-groq python-dotenv --quiet

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")
assert os.getenv("GROQ_API_KEY"), "Add GROQ_API_KEY to your .env file"
print("API key loaded.")

## Step 2 — Define the State

The state tracks what work has been done so the supervisor  
can see what's complete and what still needs doing.

In [ ]:
from typing import TypedDict, List, Optional


class ReportState(TypedDict):
    task: str                       # The user's original task
    raw_data: str                   # Analyst's output
    summary: str                    # Summarizer's output
    fact_check_notes: str           # Fact-checker's output
    final_report: str               # Compiled final output
    completed_workers: List[str]    # Which workers have run
    next_worker: str                # Supervisor's routing decision


print("State defined.")

## Step 3 — Define the Three Worker Agents

Each worker:
1. Does its specific job using the state
2. Adds itself to `completed_workers`
3. Returns to the supervisor

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatGroq(model="llama3-8b-8192", temperature=0.5)


def analyst_worker(state: ReportState) -> dict:
    """Analyse the task and extract structured data."""
    print("  [Analyst] Analysing...")

    prompt = (
        f"Analyse this task and extract key data points, metrics, and findings.\n"
        f"Be structured and specific. Use bullet points.\n\n"
        f"Task: {state['task']}"
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    print("  [Analyst] Done.")
    return {
        "raw_data": response.content,
        "completed_workers": state["completed_workers"] + ["analyst"],
    }


def summarizer_worker(state: ReportState) -> dict:
    """Summarise the analyst's findings into clear insights."""
    print("  [Summarizer] Summarising...")

    context = state["raw_data"] or state["task"]
    prompt = (
        f"Summarise the following analysis into 3-5 clear, actionable insights.\n"
        f"Each insight should be one sentence.\n\n"
        f"Analysis:\n{context}"
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    print("  [Summarizer] Done.")
    return {
        "summary": response.content,
        "completed_workers": state["completed_workers"] + ["summarizer"],
    }


def fact_checker_worker(state: ReportState) -> dict:
    """Review the summary for accuracy and flag uncertain claims."""
    print("  [Fact-Checker] Checking...")

    content = state["summary"] or state["raw_data"] or state["task"]
    prompt = (
        f"Review this content and:\n"
        f"1. Flag any claims that seem uncertain or need verification"
        f" (prefix with ⚠️)"
        f"\n2. Confirm claims that are likely accurate (prefix with ✓)\n"
        f"3. Suggest one additional fact that would strengthen the analysis\n\n"
        f"Content:\n{content}"
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    print("  [Fact-Checker] Done.")
    return {
        "fact_check_notes": response.content,
        "completed_workers": state["completed_workers"] + ["fact_checker"],
    }


print("Three workers defined: analyst, summarizer, fact_checker")

## Step 4 — Define the Supervisor

The supervisor uses the LLM to **decide who should work next** based on what's done.

It returns a `next_worker` decision:  
`analyst` | `summarizer` | `fact_checker` | `compile_report`

**Rule of thumb for the supervisor:**
1. No raw data yet → send to analyst first
2. Have data but no summary → send to summarizer
3. Have summary but no fact check → send to fact_checker
4. All three done → compile report

In [ ]:
import json
import re

SUPERVISOR_SYSTEM = """You are a supervisor managing a team of three workers:
- analyst: analyses tasks and extracts data
- summarizer: summarises findings into insights  
- fact_checker: reviews content for accuracy
- compile_report: assembles the final report (use this when ALL THREE workers are done)

Decide who should work next based on what's already been completed.
Respond with ONLY a JSON object like: {"next_worker": "analyst"}
No explanation, no extra text."""


def supervisor_node(state: ReportState) -> dict:
    """Decide which worker runs next."""
    done = state["completed_workers"]
    print(f"  [Supervisor] Completed so far: {done}")

    prompt = (
        f"Task: {state['task']}\n"
        f"Completed workers: {done}\n"
        f"Has raw data: {'yes' if state['raw_data'] else 'no'}\n"
        f"Has summary: {'yes' if state['summary'] else 'no'}\n"
        f"Has fact check: {'yes' if state['fact_check_notes'] else 'no'}\n\n"
        f"Who should work next?"
    )

    messages = [
        SystemMessage(content=SUPERVISOR_SYSTEM),
        HumanMessage(content=prompt),
    ]
    response = llm.invoke(messages)

    # Parse the JSON response
    try:
        # Find JSON in the response
        json_match = re.search(r'\{.*?\}', response.content, re.DOTALL)
        if json_match:
            decision = json.loads(json_match.group())
            next_worker = decision.get("next_worker", "compile_report")
        else:
            next_worker = "compile_report"
    except json.JSONDecodeError:
        next_worker = "compile_report"

    print(f"  [Supervisor] Routing to: {next_worker}")
    return {"next_worker": next_worker}


def compile_report_node(state: ReportState) -> dict:
    """Assemble all worker outputs into a final report."""
    print("  [Compiler] Building final report...")

    prompt = (
        f"Create a concise professional report from these inputs:\n\n"
        f"TASK:\n{state['task']}\n\n"
        f"ANALYSIS:\n{state['raw_data']}\n\n"
        f"KEY INSIGHTS:\n{state['summary']}\n\n"
        f"FACT CHECK NOTES:\n{state['fact_check_notes']}\n\n"
        f"Format as: Executive Summary, Key Findings, Caveats, Recommendations."
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    print("  [Compiler] Report ready.")
    return {"final_report": response.content}


print("Supervisor and compiler defined.")

## Step 5 — Build the Graph

The routing function reads `next_worker` from state to decide where to go.

In [ ]:
from langgraph.graph import StateGraph, START, END


def route_by_supervisor(state: ReportState) -> str:
    """Read the supervisor's decision and route accordingly."""
    return state["next_worker"]


graph = StateGraph(ReportState)

# Register all nodes
graph.add_node("supervisor",      supervisor_node)
graph.add_node("analyst",         analyst_worker)
graph.add_node("summarizer",      summarizer_worker)
graph.add_node("fact_checker",    fact_checker_worker)
graph.add_node("compile_report",  compile_report_node)

# Entry: always start with the supervisor
graph.add_edge(START, "supervisor")

# Supervisor routes dynamically to any worker
graph.add_conditional_edges(
    "supervisor",
    route_by_supervisor,
    {
        "analyst":        "analyst",
        "summarizer":     "summarizer",
        "fact_checker":   "fact_checker",
        "compile_report": "compile_report",
    },
)

# Every worker returns to the supervisor when done
graph.add_edge("analyst",       "supervisor")
graph.add_edge("summarizer",    "supervisor")
graph.add_edge("fact_checker",  "supervisor")

# compile_report is the terminal node
graph.add_edge("compile_report", END)

system = graph.compile()
print("Supervisor system compiled.")

## Step 6 — Run the System

In [ ]:
initial_state: ReportState = {
    "task": "Analyse the potential impact of large language models on the job market over the next 5 years",
    "raw_data": "",
    "summary": "",
    "fact_check_notes": "",
    "final_report": "",
    "completed_workers": [],
    "next_worker": "",
}

print(f"Task: {initial_state['task']}")
print("=" * 60)
print()

result = system.invoke(initial_state)

print("\n" + "=" * 60)
print("FINAL REPORT")
print("=" * 60)
print(result["final_report"])
print(f"\nWorkers used (in order): {result['completed_workers']}")

## Step 7 — Inspect Each Worker's Output

In [ ]:
print("ANALYST OUTPUT")
print("-" * 40)
print(result["raw_data"][:500], "...")

print("\nSUMMARIZER OUTPUT")
print("-" * 40)
print(result["summary"])

print("\nFACT-CHECKER NOTES")
print("-" * 40)
print(result["fact_check_notes"])

## Step 8 — Try a Different Task

In [ ]:
def run_supervisor_system(task: str) -> str:
    """Run the full supervisor system for any task."""
    state: ReportState = {
        "task": task,
        "raw_data": "",
        "summary": "",
        "fact_check_notes": "",
        "final_report": "",
        "completed_workers": [],
        "next_worker": "",
    }
    result = system.invoke(state)
    return result["final_report"]


# Try your own task here
report = run_supervisor_system("Evaluate the pros and cons of remote work for software companies")
print(report)

## Step 9 — Visualise

In [ ]:
from IPython.display import Image, display

try:
    display(Image(system.get_graph().draw_mermaid_png()))
except Exception:
    print(system.get_graph().draw_mermaid())

## Summary

| What you built | What it taught |
|---------------|----------------|
| Supervisor node | One LLM controls the routing for the whole team |
| Workers → supervisor loop | Workers always yield control back to the supervisor |
| `next_worker` JSON output | Structured output makes routing reliable |
| `compile_report` terminal | One node assembles everything and exits the graph |

**Supervisor vs sequential pipeline:**
| | Sequential (03) | Supervisor (04) |
|-|----------------|----------------|
| Order | Fixed | Dynamic |
| Flexibility | Low | High |
| Use when | Steps are always the same | Steps depend on the task |

---

**Next →** [05 — Collaborative Pipeline](../05-collaborative-pipeline/collaborative_pipeline.ipynb)  
A 5-agent real-world pipeline: Planner → Researcher → Analyst → Writer → Critic.